In [1]:
import pickle
import json

from boolmore.core.model import Model
from boolmore.io.load import import_phenotypes, check_phenotypes
from boolmore.algo.inference import get_phenotype_prediction
from boolmore.eval.score import get_phenotype_scores
from boolmore.io.export import export_phenotype_results
from boolmore.core.conversions import prime2bnet, prime2rr

In [2]:
json_file = "T_cell/Tcell_config.json"
EXP_FILE = "T_cell/Tcell_data.csv"

# INPUT_CACHE = "T_cell/Tcell_primes_ultrametric.pkl"
INPUT_CACHE = "T_cell/generated_models/20260825/Tcell_9820_gen94.pkl"

BASE_CACHE = "T_cell/Tcell_primes_ultrametric.pkl"

# OUTPUT_CSV = "T_cell/Tcell_base_results.csv"
OUTPUT_CSV = "T_cell/generated_models/20260825/Tcell_9820_gen94.csv"

In [3]:
with open(INPUT_CACHE, "rb") as f:
    primes = pickle.load(f)
print("Loaded primes from cache.")

Loaded primes from cache.


In [ ]:
experiments = import_phenotypes(EXP_FILE)

print(len(experiments))
print(experiments[0])

80
PhenotypeExperiment(id=1, perturbation=(), sources=(('APC', 0), ('DLL1', 0), ('IFNA_e', 0), ('IFNG_e', 0), ('IL10_e', 0), ('IL15_e', 0), ('IL1_e', 0), ('IL21_e', 0), ('IL23_e', 0), ('IL25_e', 0), ('IL27_e', 0), ('IL29_e', 0), ('IL2_e', 0), ('IL33_e', 0), ('IL36_e', 0), ('IL4_e', 0), ('IL6_e', 0), ('IL7_e', 0), ('TGFB_e', 0)), phenotype=(('GATA3', 0), ('IFNG', 0), ('IL4', 0), ('TBET', 0)), expected_exists=True, weight=1.0)


In [5]:
check_phenotypes(primes, experiments)

In [6]:
f = open(json_file)
json_dict = json.load(f)

CONSTRAINTS = json_dict["constraints"]

model = Model.import_model(primes, constraints=CONSTRAINTS)

model.check_constraint()

True

In [ ]:
predictions = get_phenotype_prediction(primes, experiments)

print(len(predictions))
print(predictions[0])

In [ ]:
score_items = get_phenotype_scores(experiments, predictions)

print(len(score_items))
print(score_items[0])

EvaluationItemScore(id=1, weight=1.0, agreement=1.0, score=1.0)


In [9]:
export_phenotype_results(experiments, predictions, score_items, OUTPUT_CSV)

id | perturbation           | sources                                                                                                                                                                                                                                                                              | phenotype                                                                                                                                                                        | expected_exists | predicted_exists | agreement | weight | score | found_phenotypes                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [10]:
with open(BASE_CACHE, "rb") as f:
    base_primes = pickle.load(f)
print("Loaded primes from cache.")

print("\n-----comparing with the baseline functions-----")
modified = 0
for node in primes:
    if prime2rr(primes[node])[1] != prime2rr(base_primes[node])[1]:
        modified += 1
        print("from_base:" + prime2bnet(node, base_primes[node]))
        print("from_ga:" + prime2bnet(node, primes[node]))

print(f"\n{modified} out of {len(primes)} functions differ from the ga results")

Loaded primes from cache.

-----comparing with the baseline functions-----
from_base:BCL6,	STAT1 & STAT1_2 & STAT3 & STAT4 & !STAT5 & !TBET & !TBET_2 & !TGFB
from_ga:BCL6,	STAT1 & STAT1_2 & STAT3 & !TBET_2 | STAT1 & STAT3 & STAT4 & !STAT5 & !TBET & !TBET_2 | STAT1 & STAT3 & !STAT5 & !TBET & !TBET_2 & !TGFB | STAT1 & STAT4 & !TBET & !TBET_2 & !TGFB
from_base:CD4,	NOTCH1 & !RUNX3 & THPOK
from_ga:CD4,	NOTCH1 | !RUNX3 & THPOK
from_base:CD8,	NOTCH1 & RUNX3 & !THPOK
from_ga:CD8,	RUNX3 & !THPOK
from_base:FOXP3,	NFAT & SMAD2 & SMAD3 & STAT5 & STAT5_2 & !STAT6 & !TBET & !TBET_2
from_ga:FOXP3,	NFAT & SMAD2 & SMAD3 & !TBET_2 | NFAT & SMAD2 & STAT5 & STAT5_2 & !STAT6 & !TBET & !TBET_2 | SMAD3 & !TBET
from_base:GATA3,	!BCL6 & !FOXP3 & GATA3 & IL25R & !IL29R & NFAT & !PU1 & STAT5 & STAT5_2 & STAT6 & !TBET & !TBET_2
from_ga:GATA3,	!BCL6 & !FOXP3 & GATA3 & IL25R & !IL29R & NFAT & !PU1 & STAT5_2 & !TBET | !BCL6 & !FOXP3 & GATA3 & IL25R & !IL29R & NFAT & !PU1 & !TBET_2 | !BCL6 & !FOXP3 & GATA3 & IL25R &